In [2]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
PREDSDIR   = CONFIGS['filepaths']['predictions']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS    = CONFIGS['experiments']['nn']['seeds']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
ORDER      = ['sr_med','sr_hi','nn_gauss']
SPLIT      = 'test'
NBINS      = 20
MINSAMPLES = 50
SRMED      = CONFIGS['experiments']['sr']['optimizedeqs']['sr_med']['init']
allmodels  = {**CONFIGS['experiments']['nn']['runs'],
              **CONFIGS['experiments']['sr']['optimizedeqs']}
COLORS     = {n: allmodels[n]['color']       for n in ORDER}
LABELS     = {n: allmodels[n]['description'] for n in ORDER}

In [4]:
def r2(obs, pred):
    m = np.isfinite(obs) & np.isfinite(pred)
    o, p = obs[m], pred[m]
    return 1 - np.sum((o-p)**2) / np.sum((o-o.mean())**2)

def bin1d(x, z, nbins=NBINS, minsamples=MINSAMPLES, plo=1, phi=99):
    m = np.isfinite(x) & np.isfinite(z)
    x, z = x[m], z[m]
    edges  = np.linspace(*np.percentile(x,[plo,phi]), nbins+1)
    xi     = np.clip(np.digitize(x,edges)-1, 0, nbins-1)
    counts = np.bincount(xi, minlength=nbins)
    sums   = np.bincount(xi, weights=z, minlength=nbins)
    return 0.5*(edges[:-1]+edges[1:]), np.where(counts>=minsamples, sums/counts, np.nan), counts

def bin2d(x, y, z, nbins=NBINS, minsamples=MINSAMPLES, plo=1, phi=99):
    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x, y, z = x[m], y[m], z[m]
    xe = np.linspace(*np.percentile(x,[plo,phi]), nbins+1)
    ye = np.linspace(*np.percentile(y,[plo,phi]), nbins+1)
    xi = np.clip(np.digitize(x,xe)-1, 0, nbins-1)
    yi = np.clip(np.digitize(y,ye)-1, 0, nbins-1)
    idx    = xi*nbins + yi
    counts = np.bincount(idx, minlength=nbins*nbins).reshape(nbins,nbins)
    sums   = np.bincount(idx, weights=z, minlength=nbins*nbins).reshape(nbins,nbins)
    return (0.5*(xe[:-1]+xe[1:]), 0.5*(ye[:-1]+ye[1:]),
            np.where(counts>=minsamples, sums/counts, np.nan), counts)

def tomap(flat):
    return np.nanmean(flat.reshape(ntime,nlat,nlon), axis=0)

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.time.size,ds.lat.size,ds.lon.size
    nsig     = ds.sizes.get('sig',1)
    lat      = ds.lat.values
    lon      = ds.lon.values
    dsig     = ds.dsig.values
    time     = ds.time.values
    fields   = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                         for v in FIELDVARS], axis=1)
    surfmask = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                if 'surfmask' in ds else None)
    flat     = lambda da: (da.transpose('time','lat','lon').values.ravel() if 'time' in da.dims
                           else np.tile(da.values,(ntime,1,1)).ravel())
    obsraw,lfraw,shfraw,lhfraw,sdoraw,seraw = [flat(ds[v]) for v in ['tp','lf','shf','lhf','sdo','se']]
    months_all = np.repeat(ds.time.dt.month.values,nlat*nlon)

kernels = []
for s in NNSEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{s}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds['k'].values)
w  = fields * np.mean(kernels,axis=0)[None,:,:] * dsig[None,None,:]
ki = (w * surfmask[:,None,:] if surfmask is not None else w).sum(axis=2)
rhraw,thetaeraw,thetaestarraw = ki[:,0],ki[:,1],ki[:,2]

predraw = {}
for n in ORDER:
    with xr.open_dataset(os.path.join(PREDSDIR,f'{n}_{SPLIT}_predictions.nc')) as ds:
        da = ds['tp'].load()
    if 'seed'       in da.dims: da = da.mean('seed')
    if 'complexity' in da.dims: da = da.isel(complexity=0)
    predraw[n] = (da.reindex(lat=lat,lon=lon,method='nearest')
                    .reindex(time=time)
                    .transpose('time','lat','lon').values.ravel())

obsmap   = tomap(obsraw)
obsvar   = np.nanstd(obsraw.reshape(ntime,nlat,nlon),axis=0)
biasmaps = {n: tomap(predraw[n]-obsraw) for n in ORDER}
varmaps  = {}
for n in ORDER:
    d = (predraw[n]-obsraw).reshape(ntime,nlat,nlon)
    varmaps[n] = np.nanmean(d**2,axis=0) - np.nanmean(d,axis=0)**2
sdomap,semap = tomap(sdoraw),tomap(seraw)

valid = np.isfinite(obsraw) & np.isfinite(rhraw) & np.isfinite(thetaeraw) & np.isfinite(thetaestarraw)
for p in predraw.values(): valid &= np.isfinite(p)

obs,rh,thetae,thetaestar = obsraw[valid],rhraw[valid],thetaeraw[valid],thetaestarraw[valid]
lf,shf,lhf,sdo,se       = lfraw[valid],shfraw[valid],lhfraw[valid],sdoraw[valid],seraw[valid]
months  = months_all[valid]
lats    = np.tile(lat[None,:,None],(ntime,1,nlon)).ravel()[valid]
lons    = np.tile(lon[None,None,:],(ntime,nlat,1)).ravel()[valid]
MODELPRED = {n: predraw[n][valid] for n in ORDER}

FEATURES = {
    'rh':         (rh,         r'$\widehat{\mathrm{RH}}$ (%)'),
    'thetae':     (thetae,     r'$\widehat{\theta_e}$ (K)'),
    'thetaestar': (thetaestar, r'$\widehat{\theta_e^*}$ (K)'),
    'lf':         (lf,         'LF (0–1)'),
    'shf':        (shf,        r'SHF (W m$^{-2}$)'),
    'lhf':        (lhf,        r'LHF (W m$^{-2}$)'),
    'sdo':        (sdo,        'SDO (m)'),
    'se':         (se,         'SE (m)')}

regions = {
    'Ocean':   lf < 0.1,
    'Land':    lf > 0.9,
    '5–15°N':  (lats>=5)  & (lats<15),
    '15–25°N': (lats>=15) & (lats<=25),
    '60–75°E': (lons>=60) & (lons<75),
    '75–90°E': (lons>=75) & (lons<=90)}

R2 = {}
for name in ORDER:
    pred = MODELPRED[name]
    R2[name] = {'overall': r2(obs,pred)}
    for m in [6,7,8]:               R2[name][m]  = r2(obs[months==m],pred[months==m])
    for rn,mask in regions.items(): R2[name][rn] = r2(obs[mask],pred[mask])

featcorr = {}
for name in ORDER:
    res = obs - MODELPRED[name]
    featcorr[name] = {}
    for f,(arr,_) in FEATURES.items():
        m = np.isfinite(arr) & np.isfinite(res)
        featcorr[name][f] = np.corrcoef(arr[m],res[m])[0,1]

gapadd,gaplog,truthgap = (MODELPRED['nn_gauss']-MODELPRED['sr_hi'],
                           np.log1p(MODELPRED['nn_gauss'])-np.log1p(MODELPRED['sr_hi']),
                           obs-MODELPRED['sr_hi'])
gaps    = {'NN − SR (add)':gapadd,'NN − SR (log)':gaplog,'Obs − SR (add)':truthgap}
gapcorr = {}
for gname,gap in gaps.items():
    gapcorr[gname] = {}
    for f,(arr,_) in FEATURES.items():
        m = np.isfinite(arr) & np.isfinite(gap)
        gapcorr[gname][f] = np.corrcoef(arr[m],gap[m])[0,1]

print(f'Loaded: {sorted(MODELPRED.keys())}')
for name in ORDER:
    print(f'  {LABELS[name]:20s}  R²={R2[name]["overall"]:.3f}')

In [ ]:
clrs = [COLORS[n] for n in ORDER]
fig,axs = pplt.subplots(nrows=1,ncols=3,refwidth=2.8,refheight=2,sharex=False,spany=True)
axs.format(ylabel=r'$R^2$',ylim=(0,0.6),yticks=0.1)

axs[0].bar(np.arange(len(ORDER)),[R2[n]['overall'] for n in ORDER],color=clrs,alpha=0.85)
axs[0].format(xticks=np.arange(len(ORDER)),xticklabels=[LABELS[n] for n in ORDER],
              xrotation=15,title=r'Overall $R^2$',grid=False)

for name in ORDER:
    axs[1].plot([6,7,8],[R2[name][m] for m in [6,7,8]],color=COLORS[name],
                linewidth=1.5,marker='o',markersize=4,label=LABELS[name])
axs[1].format(xticks=[6,7,8],xticklabels=['Jun','Jul','Aug'],title=r'$R^2$ by Month',grid=False)
axs[1].legend(loc='b',ncols=len(ORDER))

xreg  = np.arange(len(regions))
width = 0.8/len(ORDER)
for i,name in enumerate(ORDER):
    offset = (i-(len(ORDER)-1)/2)*width
    axs[2].bar(xreg+offset,[R2[name][rn] for rn in regions],
               width=width,color=COLORS[name],alpha=0.85)
axs[2].format(xticks=xreg,xticklabels=list(regions.keys()),xrotation=20,
              title=r'$R^2$ by Region',grid=False)
pplt.show()

In [ ]:
kwmap    = dict(coast=True,latlim=LATRANGE,lonlim=LONRANGE,
               latlines=[10,15,20],lonlines=[65,75,85],grid=False)
ncols    = 1+len(ORDER)
vmax_var = max(float(np.nanpercentile(varmaps[n],98)) for n in ORDER)

fig,axs = pplt.subplots(nrows=2,ncols=ncols,proj='cyl',refwidth=2,share=False)

mc = axs[0,0].pcolormesh(lon,lat,obsmap,cmap='Blues',vmin=0,vmax=2,extend='max')
axs[0,0].format(title='Observed',latlabels='l',lonlabels=False,**kwmap)
for i,name in enumerate(ORDER):
    mb = axs[0,i+1].pcolormesh(lon,lat,biasmaps[name],cmap='DryWet',vmin=-0.5,vmax=0.5,extend='both')
    axs[0,i+1].format(title=LABELS[name],latlabels=False,lonlabels=False,**kwmap)

axs[1,0].pcolormesh(lon,lat,obsvar,cmap='Purples',vmin=0,vmax=vmax_var,extend='max')
axs[1,0].format(title=r'Obs $\sigma$',latlabels='l',lonlabels='b',**kwmap)
for i,name in enumerate(ORDER):
    mv = axs[1,i+1].pcolormesh(lon,lat,varmaps[name],cmap='Purples',vmin=0,vmax=vmax_var,extend='max')
    axs[1,i+1].format(title=LABELS[name],latlabels=False,lonlabels='b',**kwmap)

fig.colorbar(mc,loc='b',col=1,label='Precipitation (mm)')
fig.colorbar(mb,loc='b',col=(2,ncols),label='Bias: Pred − Obs (mm)')
fig.colorbar(mv,loc='b',label=r'Variance (mm$^2$)')
fig.format(suptitle='Spatial Bias and Variance',rowlabels=['Bias','Variance'])
pplt.show()

In [ ]:
feats  = list(FEATURES.items())
ncols  = min(3,len(feats)); nrows = -(-len(feats)//ncols)
maxcnt = max(bin1d(arr,obs)[2].max() for _,(arr,_) in feats)

fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,refwidth=1.5,sharex=False,sharey=True)
axsf    = np.atleast_1d(axs).ravel()
for i,(ax,(fname,(arr,label))) in enumerate(zip(axsf,feats)):
    xc,obsbin,cnt = bin1d(arr,obs)
    dax = ax.twinx()
    dax.bar(xc,cnt,width=xc[1]-xc[0],absolute_width=True,edgecolor='none',color='gray5',alpha=0.3)
    dax.format(ylim=(0,1.5*maxcnt),yformatter='sci')
    if i%ncols==ncols-1 or i==len(feats)-1:
        dax.format(ylabel='Sample count')
    else:
        dax.tick_params(axis='y',labelright=False)
    ax.plot(xc,obsbin,color='k',linewidth=2,zorder=5)
    for name in ORDER:
        _,predbin,_ = bin1d(arr,MODELPRED[name])
        ax.plot(xc,predbin,color=COLORS[name],linewidth=1.5)
    ax.format(grid=False,xlabel=label)
for ax in axsf[len(feats):]: ax.set_visible(False)
axsf[0].format(ylabel='Total Precipitation (mm)')
handles  = [Line2D([],[],color='k',lw=2,label='Observed')]
handles += [Line2D([],[],color=COLORS[n],lw=1.5,label=LABELS[n]) for n in ORDER]
fig.legend(handles,loc='b',ncols=len(handles))
pplt.show()

for px,py in [('thetae','thetaestar'),('rh','thetaestar'),('rh','thetae')]:
    xarr,xl = FEATURES[px]
    yarr,yl = FEATURES[py]
    xc,yc,obsbin,obsc = bin2d(xarr,yarr,obs)
    panels = ['obs']+ORDER
    fig,axs = pplt.subplots(nrows=1,ncols=len(panels),refwidth=1.5,share=True)
    axsf    = np.atleast_1d(axs).ravel()
    for ax,panel in zip(axsf,panels):
        if panel=='obs':
            zb = obsbin; cmap,vmin,vmax,extend = 'Blues',0,3,'max'
        else:
            _,_,predbin,_ = bin2d(xarr,yarr,MODELPRED[panel])
            zb = predbin-obsbin; cmap,vmin,vmax,extend = 'DryWet',-2,2,'both'
        m = ax.pcolormesh(xc,yc,zb.T,cmap=cmap,vmin=vmin,vmax=vmax,levels=21,extend=extend)
        ax.contour(xc,yc,obsc.T,color='red3',linewidth=0.5)
        ax.format(title='Observed' if panel=='obs' else LABELS[panel],xlabel=xl,grid=False)
    axsf[0].format(ylabel=yl)
    fig.colorbar(m,loc='r',label='Model − Observed (mm)')
    pplt.show()

In [ ]:
M = rh
I = thetae - SRMED['b']*thetaestar - SRMED['c']

Mlo,Mhi = float(np.percentile(M,2)),float(np.percentile(M,98))
Ilo,Ihi = float(np.percentile(I,2)),float(np.percentile(I,98))
axlo,axhi = min(Mlo,Ilo),max(Mhi,Ihi)

panels = [('Observed',obs)] + [(LABELS[n],MODELPRED[n]) for n in ORDER]
fig,axs = pplt.subplots(nrows=1,ncols=len(panels),refwidth=2,share=True)
for i,(ax,(title,z)) in enumerate(zip(np.atleast_1d(axs).ravel(),panels)):
    xc,yc,zbin,_ = bin2d(M,I,z,nbins=35,minsamples=20,plo=2,phi=98)
    m = ax.pcolormesh(xc,yc,zbin.T,cmap='ColdHot_r',cmap_kw={'left':0.5},
                      vmin=0,vmax=4,levels=18,extend='max')
    ax.plot([axlo,axhi],[axlo,axhi],'k--',lw=1)
    ax.format(title=title,xlabel=r'$M$ (moisture)',
              ylabel=r'$I$ (instability)' if i==0 else '',
              grid=False,xlim=(Mlo,Mhi),ylim=(Ilo,Ihi))
fig.colorbar(m,loc='r',label='Precipitation (mm)')
pplt.show()

In [ ]:
kwmap = dict(coast=True,latlim=LATRANGE,lonlim=LONRANGE,
             latlines=[10,15,20],lonlines=[65,75,85],grid=False)

fig,axs = pplt.subplots(nrows=2,ncols=2,proj={1:'cyl',2:'cyl'},refwidth=3,share=False)

m = axs[0,0].pcolormesh(lon,lat,sdomap,cmap='terrain',vmin=0,vmax=900,levels=30,extend='max')
axs[0,0].format(title='SDO (m)',latlabels='l',lonlabels='b',**kwmap)
axs[0,1].pcolormesh(lon,lat,semap,cmap='terrain',vmin=0,vmax=900,levels=30,extend='max')
axs[0,1].format(title='SE (m)',latlabels=False,lonlabels='b',**kwmap)
fig.colorbar(m,loc='b',label='Elevation (m)')

for j,(oroglabel,orog) in enumerate([('SDO (m)',sdo),('SE (m)',se)]):
    lo,hi = np.nanpercentile(orog,[2,98])
    edges = np.linspace(lo,hi,11); xc = 0.5*(edges[:-1]+edges[1:])
    idxs  = np.clip(np.digitize(orog,edges)-1,0,9)
    for name in ORDER:
        r2s = [r2(obs[idxs==i],MODELPRED[name][idxs==i]) if (idxs==i).sum()>=50 else np.nan
               for i in range(10)]
        axs[1,j].plot(xc,r2s,color=COLORS[name],linewidth=1.5,marker='o',markersize=3,label=LABELS[name])
    axs[1,j].format(xlabel=oroglabel,ylabel=r'$R^2$' if j==0 else '',grid=False)
axs[1,0].legend(loc='ll',ncols=1)
pplt.show()

In [ ]:
ranked = sorted(FEATURES.keys(), key=lambda f: max(abs(featcorr[n][f]) for n in ORDER), reverse=True)
x      = np.arange(len(ranked))
width  = 0.8/len(ORDER)

fig,ax = pplt.subplots(refwidth=4,refheight=3)
for i,name in enumerate(ORDER):
    vals   = [featcorr[name][f] for f in ranked]
    offset = (i-(len(ORDER)-1)/2)*width
    ax.barh(x+offset,vals,height=width,
            color=COLORS[name],alpha=0.85-0.15*i,label=LABELS[name])
ax.format(yticks=x,
          yticklabels=[FEATURES[f][1] for f in ranked],
          xlabel='Pearson r (feature vs. residual)',
          title='Feature–Residual Correlation',grid=False,
          ylim=(-0.5,len(ranked)-0.5))
ax.axvline(0,color='gray',linewidth=0.7)
ax.legend(loc='lr',ncols=1,frame=False)
pplt.show()

In [ ]:
ranked     = sorted(gapcorr['NN − SR (add)'].keys(),
                    key=lambda f: abs(gapcorr['NN − SR (add)'][f]), reverse=True)
gap_names  = list(gaps.keys())
gap_colors = ['#2355a1','#539ed4','#D42028']
x     = np.arange(len(ranked))
width = 0.8/len(gap_names)

fig,ax = pplt.subplots(refwidth=4,refheight=3)
for gi,gn in enumerate(gap_names):
    vals   = [gapcorr[gn].get(f,0) for f in ranked]
    offset = (gi-(len(gap_names)-1)/2)*width
    ax.barh(x+offset,vals,height=width,color=gap_colors[gi],label=gn)
ax.format(yticks=x,
          yticklabels=[FEATURES[f][1] for f in ranked],
          xlabel='Pearson r',
          title='Feature Correlation with Distillation Gap',grid=False,
          ylim=(-0.5,len(ranked)-0.5))
ax.axvline(0,color='gray',linewidth=0.7)
ax.legend(loc='lr',ncols=1,frame=False)
pplt.show()

top_feats = ranked[:6]
ncols_g = min(3,len(top_feats)); nrows_g = -(-len(top_feats)//ncols_g)
fig,axs = pplt.subplots(nrows=nrows_g,ncols=ncols_g,refwidth=1.8,
                         refheight=1.5,sharex=False,sharey=True)
axsf = np.atleast_1d(axs).ravel()
for i,(ax,f) in enumerate(zip(axsf,top_feats)):
    arr,label = FEATURES[f]
    ax.axhline(0,color='gray',linewidth=0.7,linestyle='--')
    xc,means,_   = bin1d(arr,gapadd)
    ax.plot(xc,means,color='#2355a1',linewidth=1.5)
    xc_l,means_l,_ = bin1d(arr,gaplog)
    ax.plot(xc_l,means_l,color='#539ed4',linewidth=1.5,linestyle='--')
    ax.format(grid=False,xlabel=label,ylabel='Gap' if i%ncols_g==0 else '')
for ax in axsf[len(top_feats):]: ax.set_visible(False)
handles = [Line2D([],[],color='#2355a1',linewidth=1.5,label='Additive'),
           Line2D([],[],color='#539ed4',linewidth=1.5,linestyle='--',label='Log-space')]
fig.legend(handles,loc='b',ncols=2)
fig.format(suptitle='Distillation Gap vs. Features')
pplt.show()